# Case Study: Population modeling with a shared prior cache

One of the main motivations for developing [The Joker](https://thejoker.readthedocs.io) and the methods implemented in `harv` was to enable population-level inference of binary-star properties from large spectroscopic surveys with heterogenous time sampling. In this tutorial, we demonstrate how to use `harv` to perform a _hierarchical inference_ of population properties (e.g., the eccentricity distribution, the overall binary fraction) for a population of 100 simulated binary-star systems. We will:

0. Load radial velocity data for 100 simulated binary-star systems drawn from an input eccentricity distribution with a known binary fraction.
1. Generate a prior sample cache on disk using `make_prior_cache(...)` so the prior samples are generated once and reused across all stars.
2. Run the rejection sampler on every source using `RejectionSampler.run_with_samples(..., prior_cache_path)`. Because each star has the same number of epochs, the JIT cache is hit after the first call and per-star runtime drops for subsequent calls. 
3. Infer the parameters of the eccentricity distribution using the importance-sampling trick of [Hogg et al. (2010)](https://arxiv.org/abs/1008.4146).
4. Infer the "close binary fraction" using a similar importance-sampling approach.

> **Note:** We assume every star has the same number of observation epochs. This lets JAX reuse the JIT cache across all 100 rejection-sampler calls. For real surveys with heterogeneous epoch counts, see [Caveats and TODOs](#Caveats-and-TODOs) at the end for some tips.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

In [ ]:
import numpyro

numpyro.set_host_device_count(4)

In [ ]:
import copy
import pathlib
import time

import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import numpyro.distributions as dist
import scipy.optimize as opt
from population_binary_fraction_helpers import load_population
from unxt import Q, ustrip

import harv
from harv.kepler import masses

%matplotlib inline

## Load the (simulated) data

We load simulated stellar radial velocity data for 100 sources. The simulator is run separately and the data are cached as a single HDF5 file with arrays of shape `(N_stars, N_epochs)` for `time`, `rv`, and `rv_err`, plus a small table of truth values for the simulated input parameters. 

In [ ]:
DATA_PATH = pathlib.Path()


datasets, truths = load_population("../data/synthetic_binary_population/population.h5")
N_stars = len(datasets)
N_epochs = int(datasets[0].time.shape[0])
print(f"Loaded {N_stars} sources with {N_epochs} epochs per source.")

### Quick look at the data

Sanity check: a few example RV curves:

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, layout="constrained")

rng = np.random.default_rng(42)
for ax, idx in zip(axes.flat, rng.choice(N_stars, size=6, replace=False), strict=True):
    d = datasets[int(idx)]
    d.plot(ax=ax, add_labels=False)
    ax.set_title(f"star {int(idx)}", fontsize=9)

for ax in axes[1]:
    ax.set_xlabel("time [days]")
for ax in axes[:, 0]:
    ax.set_ylabel("RV [km/s]")

## Build the prior cache

The prior cache is a single HDF5 file containing prior samples in orbital parameters and any other explicitly-sampled (or nonlinear) parameters used in the model you are using to represent the data. Here, we will use a standard orbital model that assumes that the RV measurements correspond to the spectral lines from one of the stars in a binary-star system. 

We'll also use the standard parameterization of the orbital model from {class}`harv.models.StandardRV`, which uses a log-uniform prior in period and uniform priors on the angular parameters. We'll adopt a truncated normal distribution for eccentricity. You can read more about the default prior in {meth}`harv.models.StandardRV.default_prior`.

In [ ]:
prior = harv.models.StandardRV().default_prior(
    period_min=Q(0.5, "day"),
    period_max=Q(1e4, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    eccentricity=dist.TruncatedNormal(0.1, 0.25, low=0.0, high=1.0),
)
model = harv.models.RVModel()

In [ ]:
prior_cache_path = pathlib.Path("population-cache/population-prior-cache.h5")
prior_cache_path.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
harv.make_prior_cache(
    prior,
    model,
    n_samples=10_000_000,  # generate
    filename=prior_cache_path,
    key=jr.key(0),
    batch_size=1_000_000,  # You can adjust this
    return_logprobs=True,
)
print(f"Built prior cache in {time.time() - t0:.1f}s at {prior_cache_path!s}")

Now we inspect the prior cache. A prior-sample file is just a serialized {class}`harv.samplers.Samples` file but with empty linear parameters, so {meth}`harv.samplers.Samples.from_hdf5` loads it directly.

In [ ]:
prior_cache_samples = harv.Samples.from_hdf5(prior_cache_path)
print(f"n_samples = {prior_cache_samples.n_samples}")
print(f"nonlinear keys: {sorted(prior_cache_samples.nonlinear)}")

## Run rejection sampling on all 100 sources

Now we run the rejection sampler on all of the sources in our dataset. We use the {meth}`harv.samplers.RejectionSampler.run_with_samples` method, which differs from the standard `run` method in that it takes either an in-memory `Samples` object as input, or the path to a prior cache file instead of generating new samples on the fly. This allows us to reuse the same prior samples across all 100 sources, which is much faster than generating new samples for each source.

Each call to `sampler.run_with_samples(data, PRIOR_CACHE_FILENAME)` streams the prior cache from disk in `batch_size`-row chunks. The expensive computation is evaluating the model (marginal) log-likelihoods for all of the prior samples with the data for a given star. Because the model is JIT-compiled, the first call to `run_with_samples` will be slow as the JIT cache is built, but subsequent calls will be faster because the JIT cache is hit. 

We'll store the posterior samples for each star in a separate HDF5 file in the `population-cache/population-posteriors` directory:

In [ ]:
posteriors_path = pathlib.Path("population-cache/posteriors")
posteriors_path.mkdir(parents=True, exist_ok=True)

We also need to specify how many posterior samples to retain per star. If your population model is simple, you probably don't need more than a few hundred posterior samples per star to get a good estimate of the population parameters. If your population model is more complex, you may need more posterior samples per star to get a good estimate of the population parameters. Here we adopt 512, but you should increase this substantially if your population model has more parameters or is more complex.

In [ ]:
max_post_samples = 512

In [ ]:
sampler = harv.RejectionSampler(prior, model, batch_size=1_000_000)

posteriors: list[harv.Samples] = []
timings: list[float] = []

for n, data in enumerate(datasets):
    t0 = time.time()

    # Note that we set a per-star random number seed: this is important for ensuring
    # reproducible randomness across the stars
    samples = sampler.run_with_samples(
        data,
        prior_cache_path,
        max_posterior_samples=max_post_samples,
        seed=n,
        return_logprobs=True,
    )
    dt = time.time() - t0
    timings.append(dt)
    posteriors.append(samples)
    samples.to_hdf5(posteriors_path / f"star_{n:04d}.h5")

    if n < 4 or n % 25 == 0:
        print(f"star {n:>3}: {samples.n_samples:>4} accepted in {dt:.2f}s")

print()
print(f"first call  : {timings[0]:.2f}s (JIT compile included)")
print(f"second call : {timings[1]:.2f}s")
print(f"median      : {np.median(timings):.2f}s")
print(f"total       : {sum(timings):.1f}s for {N_stars} stars")

### Per-source rejection sampling timing

Having the same `data` pytree shape across all 100 sources means the second-and-onward calls hit the JIT cache. The first-call cost is dominated by compilation; subsequent calls should be bounded by disk I/O + linear-parameter resampling:


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(timings, "o", ms=3)
ax.axhline(np.median(timings[1:]), color="C1", lw=1, label="median (warm)")
ax.set_xlabel("star index")
ax.set_ylabel("wall time [s]")
ax.set_title("Per-star rejection-sampler runtime")
ax.legend()
fig.tight_layout()

### Histogram of accepted posterior samples

A handful of stars may end up with very few accepted samples, $K_n$ (number of posterior samples $K$ for star indexed by $n$). This usually happens when the data are very constraining and so the modes of the likelihood are very narrow (i.e. the orbital solution is well-determined). In principle, if your population model over parameters like period and eccentricity is broad, follow-up hierarchical analyses can use a variable number of returned samples and should not be too impacted by having different numbers of posterior samples per star. More on this in the next cells.

In [ ]:
n_accept = np.array([s.n_samples for s in posteriors])
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(n_accept, bins=30)
ax.set_xlabel("$K_n$ (accepted samples per star)")
ax.set_ylabel("# of stars")
ax.set_title(f"Acceptance histogram (max capped at {max_post_samples})")
fig.tight_layout()

## Generate more posterior samples with MCMC

The hierarchical inference framework we introduce below uses the per-source posterior samples from the rejection sampler within the population likelihood. This involves Monte Carlo sums of the form $\frac{1}{K_n} \sum_j p(e_{nj} \mid \boldsymbol{\alpha}) / p_\mathrm{int}(e_{nj})$, so a star with very few accepted samples (small $K_n$) contributes a noisy term that can sometimes impact the gradient of the population likelihood. Two failure modes lead to small $K_n$ after rejection:

1. *A well-constrained but low-acceptance posterior*: the likelihood is sharply peaked, so almost every prior draw is rejected, but the surviving sample(s) are tightly clustered around the true orbit. 
2. *Certain multi-modal posteriors*: the rejection samples may be spread across a small number of well-separated modes. This can often happen when the data have a long time baseline, or are otherwise fairly strongly constraining of the orbit, but where some period aliasing or other degeneracies in the data prevent a unique orbital solution. 

For the first case, we can simply generate more posterior samples with MCMC, warm-started from the rejection samples. For the second case, the multi-modality can lead to poor MCMC performance and a very noisy importance-sampling sum that can cause problems for gradient-based inference of the population parameters. In this case, we can flag the star as "multi-modal" and keep it as-is in the importance-sampling sum, but with a small $K_n$ to downweight its contribution to the population likelihood.

We use `Samples.period_unimodal(data)` to distinguish the two cases. Stars below the threshold and judged unimodal are re-sampled with {class}`harv.samples.NumpyroSampler` (warm-started from the rejection samples). The multimodal cases with few returned samples are flagged and kept as-is — the importance-sampling sum still uses them, just with a small $K_n$.


In [ ]:
stars_thin = [n for n, s in enumerate(posteriors) if s.n_samples < max_post_samples]
print(f"{len(stars_thin)} stars have fewer than {max_post_samples} samples")

In [ ]:
n_warmup = 512
n_samples = 512
n_chains = 2

mcmc_sampler = harv.NumpyroSampler(prior, model)

In [ ]:
followed_up: list[int] = []
multimodal: list[int] = []
skipped_empty: list[int] = []
mcmc_failed: list[tuple[int, str]] = []

for n in stars_thin:
    samples = posteriors[n]
    data = datasets[n]

    if samples.n_samples == 0:
        skipped_empty.append(n)
        continue

    if not samples.period_unimodal(data):
        multimodal.append(n)
        continue

    t0 = time.time()
    try:
        new_samples = mcmc_sampler.run(
            data,
            init_samples=samples,
            seed=1 + n,
            num_warmup=n_warmup,
            num_samples=n_samples,
            num_chains=n_chains,
            return_logprobs=True,
        )
    except Exception as exc:  # noqa: BLE001 -- diagnostic, not silenced
        mcmc_failed.append((n, repr(exc)))
        continue
    dt = time.time() - t0

    posteriors[n] = new_samples[:max_post_samples]
    new_samples.to_hdf5(posteriors_path / f"star_{n:04d}.h5")
    followed_up.append(n)

print()
print("summary:")
print(f"  followed-up (MCMC):           {len(followed_up)}")
print(f"  thin & multimodal (kept):     {len(multimodal)}")

Now, almost all of the stars have enough posterior samples to contribute a stable term to the population likelihood. A few stars are flagged as being multi-modal with few returned samples. For those cases, we could try using a different MCMC sampling technique (e.g., Nested sampling) to generate more samples. For now, we'll just keep them as-is and accept that they contribute a noisy term to the population likelihood.

In [ ]:
n_accept_after = np.array([s.n_samples for s in posteriors])
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(0, max(n_accept_after.max(), max_post_samples) + 1, 40)
ax.hist(n_accept, bins=bins, alpha=0.5, label="rejection only")
ax.hist(n_accept_after, bins=bins, alpha=0.5, label="after MCMC top-up")
ax.axvline(max_post_samples, color="C3", lw=1, ls="--")
ax.set_xlabel("$J_n$")
ax.set_ylabel("# of stars")
ax.legend()
fig.tight_layout()

## Inferring the eccentricity distribution with a hierarchical model

We now have posterior samples for each star in our population. 
We can use these samples to infer a _hierarchical model_ of the parameters of a population-level eccentricity distribution.
First, some mathematical context.

We want to infer the parameters $\boldsymbol{\alpha}$ of a population-level eccentricity distribution $p(e \mid \boldsymbol{\alpha})$. 
For example, $\boldsymbol{\alpha}$ could represent the mean and standard deviation of a truncated normal distribution on $e$, or the shape parameters of a Beta distribution on $e$. 
We have (a finite number of) posterior samples over the per-source parameters $\boldsymbol{\theta}$. The standard hierarchical likelihood is
$$
\mathcal{L}(\{D_n\} \mid \boldsymbol{\alpha}, \boldsymbol{\theta}) = 
    \prod_n p(D_n \mid \theta_n) \, p(\theta_n \mid \boldsymbol{\alpha})
    \quad .
$$
where $p(D_n \mid \theta_n)$ is the corresponds to the likelihood of the observed radial velocity data $D_n$ for star $n$ given the orbital parameters of that star, $\theta_n$.
The term $p(\theta_n \mid \boldsymbol{\alpha})$ is the connection between the orbital parameters for that star and the population parameters $\boldsymbol{\alpha}$.
We may then be interested in the hierarchical likelihood marginalized over the per-source orbital parameters, $p(D_n \mid \alpha)$, which is
$$
\hat{\mathcal{L}}(\{D_n\} \mid \boldsymbol{\alpha}, \boldsymbol{\theta}) = 
    \prod_n \int \mathrm{d}\theta_n \, 
        p(D_n \mid \theta_n) \, p(\theta_n \mid \boldsymbol{\alpha})
    \quad .
$$

Following [Hogg et al. 2010](https://arxiv.org/abs/1008.4146) or [Price-Whelan et al. 2020](https://arxiv.org/abs/2002.00014), if we have $K_n$ posterior samples for star $n$ over the orbital parameters with sample index $k$ given by $\theta_{nk}$ generated under an *interim* prior $p_\mathrm{int}(\theta)$, the marginalization integral can be computed with a Monte Carlo sum:
$$
\int \mathrm{d}\theta_n \, p(D_n \mid \theta_n)\, p(\theta \mid \boldsymbol{\alpha})
    \,\;\approx\; 
    \frac{Z_n}{K_n} \sum_{k=1}^{K_n} 
        \frac{p(\theta_{nk} \mid \boldsymbol{\alpha})}{p_\mathrm{int}(\theta_{nk})},
$$
where $Z_n$ is the evidence for star $n$ (independent of $\boldsymbol{\alpha}$). So up to a constant in $\boldsymbol{\alpha}$:
$$
\ln \mathcal{L}(\{D_n\} \mid \boldsymbol{\alpha}) \;=\; \sum_n \ln \!\left[\frac{1}{K_n} \sum_{k=1}^{K_n} \frac{p(\theta_{nk} \mid \boldsymbol{\alpha})}{p_\mathrm{int}(\theta_{nk})}\right] + \mathrm{const}.
$$

The interim prior on $e$ is what we defined when creating the `prior` above: a truncated normal distribution. We will fit for a population-level truncated normal hyperprior on $[0, 1]$ with mean $\mu$ and stddev $\sigma$.

In [ ]:
# Stack posterior eccentricities per star into a padded matrix so the
# inner sum can be vmapped.  Pad with a sentinel and mask.
K_max = max(s.n_samples for s in posteriors)
ecc_padded = np.full((N_stars, K_max), np.nan, dtype=np.float32)
mask = np.zeros((N_stars, K_max), dtype=bool)
for n, s in enumerate(posteriors):
    j = s.n_samples
    ecc_padded[n, :j] = np.asarray(s["eccentricity"].value)
    mask[n, :j] = True

ecc_padded = jnp.asarray(ecc_padded)
mask = jnp.asarray(mask)
ln_K_n = jnp.log(mask.sum(axis=1).astype(jnp.float32))

In [ ]:
@jax.jit
def neg_ln_pop_likelihood(params: jax.Array) -> jax.Array:
    """Negative ln marginal likelihood for hyperparameters.

    ``params = (mean, ln_std)`` so the stddev stays positive under the
    unconstrained optimiser.
    """
    mean = params[0]
    std = jnp.exp(params[1])

    ln_interim_prior = prior.nonlinear_priors["eccentricity"].log_prob(ecc_padded)
    ln_pop_prior = dist.TruncatedNormal(mean, std, low=0.0, high=1.0).log_prob(
        ecc_padded
    )

    log_w = ln_pop_prior - ln_interim_prior
    log_w = jnp.where(mask, log_w, -jnp.inf)
    ln_inner = jax.scipy.special.logsumexp(log_w, axis=1) - ln_K_n
    return -ln_inner.sum()

In [ ]:
x0 = jnp.array([0.5, 0.5])
res = opt.minimize(neg_ln_pop_likelihood, x0=x0, method="Nelder-Mead")
mean_hat = float(res.x[0])
std_hat = float(np.exp(res.x[1]))
print(f"Best-fit TruncatedNormal(mean, std) = ({mean_hat:.3f}, {std_hat:.3f})")
print(f"  -ln L = {res.fun:.3f}")
if "ecc_dist_true" in truths:
    mu_t, std_t = truths["ecc_dist_true"]
    print(f"  truth = ({mu_t:.3f}, {std_t:.3f})")

### Visualise the inferred eccentricity distribution

Overlay the best-fit Beta against the input truth (if available) and the interim prior.

In [ ]:
def truncnorm_pdf(e, mean, std):
    return np.exp(
        np.asarray(dist.TruncatedNormal(mean, std, low=0.0, high=1.0).log_prob(e))
    )

In [ ]:
e_grid = np.linspace(1e-3, 1 - 1e-3, 200)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(
    e_grid,
    truncnorm_pdf(e_grid, mean_hat, std_hat),
    lw=2,
    marker="",
    label="optimized population model",
)
ax.plot(
    e_grid,
    jnp.exp(prior.nonlinear_priors["eccentricity"].log_prob(e_grid)),
    color="#aaaaaa",
    marker="",
    lw=1,
    label="interim prior",
)

ax.plot(
    e_grid,
    truncnorm_pdf(e_grid, *truths["ecc_dist_true"]),
    "tab:green",
    marker="",
    label="truth",
)

ax.set_xlabel("eccentricity, $e$")
ax.set_ylabel("$p(e)$")
ax.legend()

## Close binary fraction

We now demonstrate two ways we can use the per-source posterior samples to estimate the close binary fraction in our sample. 
We define the close binary fraction as the fraction of stars with a companion in a defined region of $(P, M_2)$ space. 
We use $P \in (1, 1000)\,\mathrm{day}$ and $M_2 > 0.1\,M_\odot$.

We first implement a simple estimator for the close binary fraction that uses the samples directly.
We then implement a more sophisticated hierarchical inference approach that models the population of companions in $(P, M_2)$ space with continuous distributions and infers the parameters of the distributions, including the close binary fraction.

Both methods rely on computing posterior samples for the companion mass, $M_2$, for each star in the sample -- this is just a transformation of the orbital parameters, but it requires an estimate of the primary mass $M_1$ and an assumption about the inclination $i$ of the orbit.

#### Companion mass, $M_2$

We estimate the corresponding companion mass $M_2$ for each posterior sample by assuming we have an independent estimate of the primary mass $M_1$ for each star, and assuming that the inclination $i$ is isotropic (but unobserved).
That is, for each posterior sample over orbital parameters $\theta_{nk} = (P, e, K)_{nk}$, we sample an $(M_1, \sin i)$ pair from the error distribution of $M_1$ and an isotropic prior for $\cos i$. We then solve for $M_{2, nj}$ using the standard mass function relation for single-lined spectroscopic binaries:
$$
\frac{(M_2 \, \sin i)^3}{(M_1 + M_2)^2} = \frac{P \, K^3}{2\pi \, G} (1 - e^2)^{3/2}
\quad .
$$ 

We draw $M_1$ samples from a Gaussian with mean and standard deviation taken from the catalog, and draw $\cos i$ from a uniform distribution on $[-1, 1]$.
{func}`harv.kepler.masses.companion_mass_from_mass_function` solves for the companion mass:

In [ ]:
# placeholder catalogue uncertainty on M_1 -- this should be taken from the catalog, but
# we just use a constant value here for demonstration
m1_err = Q(0.05, "Msun")

# mock "observe" the M_1 values by adding noise to the truth values
key = jr.key(42)
key, key_m1 = jr.split(key)

m1_pad_k = Q(
    truths["M1"][:, None]
    + ustrip("Msun", m1_err) * jr.normal(key_m1, (N_stars, K_max)),
    "Msun",
)

In [ ]:
key, key_cos = jr.split(key)
cos_i_pad = jr.uniform(key_cos, (N_stars, K_max), minval=-1.0, maxval=1.0)
sini_pad_k = jnp.sqrt(1.0 - cos_i_pad**2)

Note: if you need to compute $M_2$ for a large number of stars, you can use {func}`harv.kepler.binary_mass_function` and {func}`harv.kepler.companion_mass_from_mass_function` in a vectorized way to compute $M_2$ for all of the posterior samples across all of the stars in one go. However, you have to first pre-construct 2D arrays of the period, semi-amplitude, and eccentricity samples for all of the stars with padded arrays for stars with fewer posterior samples than the maximum across the sample. Here we just use a for loop for simplicity:

In [ ]:
posteriors_with_m2 = []
for n, s in enumerate(posteriors):
    # Enforce K > 0 first.  The rejection-sampler posterior on (K, arg_peri)
    # is bimodal -- (K, w) and (-K, w + pi) predict the same orbit.
    # ``binary_mass_function`` uses K^3 directly, so feeding negative K
    # yields a negative mass function and breaks the bisection in
    # ``companion_mass_from_mass_function`` (it returns M2 ~ 0), which
    # would silently misclassify ~50% of every binary as sub-stellar.
    s = s.wrap_angles()
    s_new = copy.deepcopy(s)

    mf = masses.binary_mass_function(s["period"], s["rv_semiamp"], s["eccentricity"])
    m2 = masses.companion_mass_from_mass_function(
        mf, m1=m1_pad_k[n, : s.n_samples], sini=sini_pad_k[n, : s.n_samples]
    )
    s_new.nonlinear["m2"] = m2
    posteriors_with_m2.append(s_new)

### The estimator approach

If all stars were edge-on ($\sin i = 1$) and we had perfect knowledge of $M_1$ and all of the orbital parameters, we could compute the binary fraction simply by counting the fraction of stars with $P$ and $M_2$ in the defined region of parameter space. 
In practice, we have to account for the uncertainty in all of these quantities, and often our constraints on $P$ and $M_2$ are quite broad or multi-modal.
A simple estimator for the close binary fraction is to compute the probability that each star has $P$ and $M_2$ in the defined region of parameter space by, conceptually, counting the fraction of posterior samples that satisfy the condition. 
That is, we define an indicator function $I_{nk}$ for each posterior sample $k$ of star $n$ that is 1 if the sample satisfies the condition and 0 otherwise:
$$
I_{nk} = \begin{cases} 1 & P_{\mathrm{min}} < P_{nk} < P_{\mathrm{max}} \text{ and } M_{2,nk} > M_{2,\mathrm{cut}} \\ 0 & \text{otherwise} \end{cases}
$$
where we defined our adopted period limits $P_\mathrm{min}$ and $P_\mathrm{max}$ and the companion mass cut $M_{2,\mathrm{cut}}$ above.

From this, we can compute the probability that star $n$ has a companion in the defined region of parameter space as the fraction of posterior samples that satisfy the condition:
$$
p_n = \frac{1}{K_n} \sum_{k=1}^{K_n} I_{nk}.
$$

Let's compute these things for each star in our sample:

In [ ]:
# Binary fraction cuts defined above
P_MIN_CUT = Q(1.0, "day")
P_MAX_CUT = Q(1000.0, "day")
M2_CUT = Q(0.1, "Msun")

In [ ]:
p_n = jnp.array(
    [
        jnp.sum(
            (s["period"] > P_MIN_CUT) & (s["period"] < P_MAX_CUT) & (s["m2"] > M2_CUT)
        )
        / s.n_samples
        for s in posteriors_with_m2
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(p_n, bins=np.linspace(0, 1, 31))
ax.set_xlabel("$p_n$ (per-star probability of being a close binary)")
ax.set_ylabel("# of stars")
fig.tight_layout()

With the per-star probabilities $p_n$ in hand, we can compute the close binary fraction by optimizing the likelihood:
$$
\mathcal{L}(f) = \prod_{n=1}^N \left[(1 - f)(1 - p_n) + f\, p_n\right].
$$

In [ ]:
def neg_log_likelihood_f(f: float, p: np.ndarray) -> float:
    val = (1.0 - f) * (1.0 - p) + f * p
    # Guard against log(0) on the boundary.
    return -np.log(np.clip(val, 1e-300, None)).sum()


res_f = opt.minimize_scalar(
    neg_log_likelihood_f, args=(p_n,), bounds=(0.0, 1.0), method="bounded"
)
f_hat = float(res_f.x)
print(f"Best-fit close binary fraction: f_hat = {f_hat:.3f}")
print(f"  truth: f = {truths['binary_fraction_true']:.3f}")

OK - we got somewhat close to the true value, but we also want to know the uncertainty on our estimate of $f$. We estimate this uncertainty with _bootstrap_ resamplings of the data (here: the per-star probabilities):

In [ ]:
N_BOOT = 1024
rng = np.random.default_rng(0)
f_boot = np.empty(N_BOOT)

for b in range(N_BOOT):
    idx = rng.integers(0, p_n.size, size=p_n.size)
    r = opt.minimize_scalar(
        neg_log_likelihood_f,
        args=(p_n[idx],),
        bounds=(0.0, 1.0),
        method="bounded",
    )
    f_boot[b] = r.x

lo, med, hi = np.percentile(f_boot, [16, 50, 84])
print(f"bootstrap percentiles (16/50/84): {lo:.2f} / {med:.2f} / {hi:.2f}")
print(f"f_hat = {f_hat:.2f} (-{f_hat - lo:.2f}, +{hi - f_hat:.2f})")

fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
ax.hist(f_boot, bins=40)
ax.axvline(f_hat, color="C1", lw=2, label=f"$\\hat f = {f_hat:.2f}$")
if "binary_fraction_true" in truths:
    ax.axvline(
        truths["binary_fraction_true"],
        color="C2",
        lw=2,
        ls="--",
        label=f"truth = {truths['binary_fraction_true']:.2f}",
    )
ax.set_xlabel("close binary fraction $f$")
ax.set_ylabel("bootstrap count")
ax.legend()

It looks like our estimate of the close binary fraction is consistent with the true value (within the bootstrap uncertainty on the binary fraction), but perhaps biased a little to lower values. We can do better by implementing a hierarchical model for the population of companions in $(P, M_2)$ space and inferring the parameters of that model. We develop this model in the next sections, but it requires quite a bit more infrastructure.

## Joint hierarchical inference of $(f, p(M_2, P, e))$

The hard-cut estimator of Section 4 deflates $\hat f$ near the $M_2$ cut boundary. The principled fix is to **drop the hard cut and jointly infer the population distributions of $M_2$, $P$, and $e$ together with $f$**, using the [Hogg–Myers–Bovy (2010)](https://arxiv.org/abs/1008.4146) importance-sampling estimator (already used in Section 3 for eccentricity alone) inside a numpyro NUTS run. This subsumes Section 3's eccentricity-only fit and removes the boundary bias because every posterior sample contributes a *continuous* weight rather than a step function.

We parameterise the binary population as

- $M_2 \sim \mathrm{LogNormal}(\mu_{\log M_2},\, \sigma_{\log M_2})$ left-truncated at a brown-dwarf cutoff $M_{2,\min}^{\rm BD} = 0.05\,M_\odot$. The truncation is essential: without it, the importance-sampling estimator finds a degenerate mode in which $p(M_2)$ widens to $M_2 \to 0$ and *singles masquerade as very-low-mass binaries*, pushing $f \to 1$.
- $e \sim \mathrm{TruncatedNormal}(\mu_e,\, \sigma_e)$ on $[0, 1]$,

and hold the **period** distribution fixed at the rejection-sampler interim prior. The simulator drew $P$ log-uniformly on a sub-range of the prior, so the period factor approximately cancels in the importance ratio (modulo the $P$-cut indicator). Inferring $p(P)$ jointly is a clean follow-up.

> **Comparing to Section 4.** The raw hyperparameter $f$ above is the fraction of stars whose orbits are drawn from the binary population with $M_2 > 0.05\,M_\odot$. Section 4 instead uses the hard cut $M_2 > 0.1\,M_\odot$. To match Section 4, we report
>
> $$ f_\mathrm{close} = f \cdot P\!\left(M_2 > 0.1\,M_\odot \mid \text{inferred LogNormal}\right), $$
>
> where the conditional probability is evaluated under the *truncated* LogNormal (renormalised by $1 - F(0.05)$). $f_\mathrm{close}$ is the apples-to-apples comparison value; the raw $f$ is informative about the underlying population.

### The importance-sampling weight

We augment the rejection-sampler parameter space $(P, e, K)$ with auxiliaries $(M_1, \sin i)$ — drawn from the catalog distribution and an isotropic inclination prior. We reuse the **same draws Section 4 used** (same `rng` seed), so the $M_2^{(nj)}$ values match exactly without re-Monte-Carlo.

The change of variables $K \leftrightarrow M_2$ at fixed $(P, e, M_1, \sin i)$ converts a population density on $(P, e, M_2)$ to one over $(P, e, K)$:

$$
p_\mathrm{pop}^{(P, e, K)}(P, e, K) = p_\mathrm{pop}^{(P, e, M_2)}(P, e, M_2(K)) \cdot \left|\frac{\partial M_2}{\partial K}\right|.
$$

Assuming the period factor cancels except through a cut indicator, the per-sample importance weight is

$$
w_{nj}(f, \boldsymbol\alpha) = \mathbb{1}[P_{nj} \in P\text{-cut}] \cdot
\frac{p_\mathrm{pop}^{M_2}(M_2^{(nj)} | \mu_{\log M_2}, \sigma_{\log M_2})}{p_\mathrm{int}^K(K_{nj} | P_{nj})} \cdot
\frac{p_\mathrm{pop}^{e}(e_{nj} | \mu_e, \sigma_e)}{p_\mathrm{int}^{e}(e_{nj})} \cdot
\left|\frac{\partial M_2}{\partial K}\right|_{nj}.
$$

The per-star average weight is $\bar w_n = J_n^{-1} \sum_j w_{nj}$. Treating "single" stars as draws from the rejection-sampler interim prior (uninformative population structure), the mixture log-likelihood is

$$
\ln p(D_n | f, \boldsymbol\phi) = \ln\!\big[f\,\bar w_n(\boldsymbol\phi) + (1 - f)\big] + \mathrm{const}.
$$

The Jacobian, from $K = (2\pi G/P)^{1/3}\,M_2 \sin i / [(M_1+M_2)^{2/3}\sqrt{1-e^2}]$,

$$
\frac{\partial K}{\partial M_2} = \frac{(2\pi G/P)^{1/3}\,\sin i}{\sqrt{1-e^2}} \cdot \frac{3M_1 + M_2}{3(M_1 + M_2)^{5/3}},
$$

is evaluated numerically with `jax.grad` for safety. We sample $(f, \mu_{\log M_2}, \sigma_{\log M_2}, \mu_e, \sigma_e)$ with numpyro NUTS.

In [ ]:
# 5.1 Per-sample log-densities and Jacobian for the importance estimator.
#
# Build padded (N_stars, K_max) arrays from posteriors_with_m2 so the
# population and interim densities + the Jacobian can be evaluated in a
# single broadcasted call.  Then compute the IS-specific log terms.

import types

from harv.kepler.constants import G

# --- Build padded arrays ---
K_max_5 = max(s.n_samples for s in posteriors_with_m2)
mask_pad_j_np = np.zeros((N_stars, K_max_5), dtype=bool)
P_pad_j_np = np.ones((N_stars, K_max_5), dtype=np.float32)
e_pad_j_np = np.full((N_stars, K_max_5), 0.5, dtype=np.float32)
K_pad_j_np = np.ones((N_stars, K_max_5), dtype=np.float32)
M2_pad_j_np = np.full((N_stars, K_max_5), 0.5, dtype=np.float32)
for n, s in enumerate(posteriors_with_m2):
    j = s.n_samples
    if j == 0:
        continue
    mask_pad_j_np[n, :j] = True
    P_pad_j_np[n, :j] = ustrip("day", s.nonlinear["period"])
    e_pad_j_np[n, :j] = s.nonlinear["eccentricity"].value
    K_pad_j_np[n, :j] = s.linear["rv_semiamp"].value
    M2_pad_j_np[n, :j] = ustrip("Msun", s.nonlinear["m2"])
P_pad_j = jnp.asarray(P_pad_j_np)
e_pad_j = jnp.asarray(e_pad_j_np)
K_pad_j = jnp.asarray(K_pad_j_np)
M2_pad_j = jnp.asarray(M2_pad_j_np)
mask_pad_j = jnp.asarray(mask_pad_j_np)
# m1_pad_k and sini_pad_k are 2D padded arrays from cells above.
M1_pad_j = ustrip("Msun", m1_pad_k)
sini_pad_j = sini_pad_k

# --- Interim prior on K | P, e (PeriodDependentKPrior callable). ---
_k_qd = prior.linear_priors["rv_semiamp"](
    types.SimpleNamespace(
        period=Q(P_pad_j, "day"),
        eccentricity=e_pad_j,
    )
)
log_pint_K_pad = _k_qd.distribution.log_prob(K_pad_j)

# --- Interim prior on e ---
log_pint_e_pad = prior.nonlinear_priors["eccentricity"].log_prob(e_pad_j)

# --- Jacobian |dM_2 / dK| via autodiff on the analytic K(M_2; P, e, M_1, sin i) ---
G_km3_Msun_d2 = float(ustrip("km**3 / (Msun * day**2)", G))


def _K_kms(M2, P_d, e, M1, sini):
    pref = (2.0 * jnp.pi * G_km3_Msun_d2 / P_d) ** (1.0 / 3.0)
    return (
        pref * M2 * sini / ((M1 + M2) ** (2.0 / 3.0) * jnp.sqrt(1.0 - e**2)) / 86400.0
    )


dK_dM2_flat = jax.vmap(jax.grad(_K_kms, argnums=0))(
    M2_pad_j.ravel(),
    P_pad_j.ravel(),
    e_pad_j.ravel(),
    M1_pad_j.ravel(),
    sini_pad_j.ravel(),
)
log_dM2_dK_pad = (-jnp.log(jnp.abs(dK_dM2_flat))).reshape(M2_pad_j.shape)

# --- Period-related precomputes (NEW: TruncN on log10 P with hyperparams). ---
# We now INFER a TruncatedNormal on log10(P) inside the cut, replacing
# the LogUniform-within-cut binary period model.  Precompute:
#   - log10_P_pad: per-sample log10(P) for the population log-prob.
#   - in_P_cut_pad: cut mask (TruncN truncation matches the cut bounds).
#   - LOG10_PRIOR_RATIO_CONST: constant in the log_pop_P - log_pint_P
#     ratio that accounts for the change of variables P <-> log10(P) and
#     the interim LogUniform normalisation.  Specifically,
#         log_pop_P(P) - log_pint_P(P)
#       = log_pop_logP(log10 P) - log(P ln 10)
#         - [-log P - log(ln(Pmax_prior/Pmin_prior))]
#       = log_pop_logP(log10 P) + log(log10(Pmax_prior/Pmin_prior)).
_pinst = prior.nonlinear_priors["period"]
P_PRIOR_MIN_DAY = float(ustrip("day", Q(float(_pinst.distribution.low), _pinst.unit)))
P_PRIOR_MAX_DAY = float(ustrip("day", Q(float(_pinst.distribution.high), _pinst.unit)))
P_CUT_MIN_DAY = float(ustrip("day", P_MIN_CUT))
P_CUT_MAX_DAY = float(ustrip("day", P_MAX_CUT))

log10_P_pad = jnp.log10(P_pad_j)
in_P_cut_pad = (P_pad_j > P_CUT_MIN_DAY) & (P_pad_j < P_CUT_MAX_DAY)
LOG10_PRIOR_RATIO_CONST = float(np.log(np.log10(P_PRIOR_MAX_DAY / P_PRIOR_MIN_DAY)))
LOG10_P_CUT_LOW = float(np.log10(P_CUT_MIN_DAY))
LOG10_P_CUT_HIGH = float(np.log10(P_CUT_MAX_DAY))

# ln(K_n) for the IS sum.  Keep the mask_j_5 / ln_J_n_5 aliases for the MCMC cell.
mask_j_5 = mask_pad_j
ln_J_n_5 = jnp.log(jnp.maximum(mask_pad_j.sum(axis=1).astype(jnp.float32), 1.0))

print(
    f"Log10 period cut: [{LOG10_P_CUT_LOW:.2f}, {LOG10_P_CUT_HIGH:.2f}]  "
    f"(P in [{P_CUT_MIN_DAY:.1f}, {P_CUT_MAX_DAY:.1f}] day)"
)
print(f"Constant in log_pop_P - log_pint_P:  {LOG10_PRIOR_RATIO_CONST:+.4f}")
print(f"Total non-padded posterior samples across stars: {int(mask_pad_j.sum())}")

In [ ]:
# 5.2 Numpyro hierarchical model + NUTS MCMC.
#
# Mixture INSIDE the importance ratio:
#   p_pop(theta) = f * p_pop_bin(theta | alpha) + (1-f) * p_pop_sing(theta | sigma_sing)
#
# Binary component:
#   - log10(P) ~ TruncatedNormal(mu_logP, sig_logP) on [log10(1), log10(1000)]
#     (NEW -- previously LogUniform inside the cut).
#   - e ~ TruncatedNormal(mu_e, sig_e) on [0, 1].
#   - log(M_2) ~ TruncatedNormal(mu_logM2, sig_logM2) on [log 0.05, log 5].
#
# Single component: HalfNormal in M_2 peaked at 0 (width sigma_sing).
# Singles inherit the rejection-sampler interim prior on P and e, so
# those factors cancel against p_int in the importance ratio.

import numpyro
from numpyro.infer import MCMC, NUTS

LOG_M2_LOW = jnp.log(0.05)
LOG_M2_HIGH = jnp.log(5.0)


def population_model():
    f = numpyro.sample("f", dist.Uniform(0.0, 1.0))
    mu_logM2 = numpyro.sample("mu_logM2", dist.Uniform(jnp.log(0.1), jnp.log(2.0)))
    sig_logM2 = numpyro.sample("sig_logM2", dist.Uniform(0.2, 1.2))
    mu_e = numpyro.sample("mu_e", dist.Uniform(0.05, 0.95))
    sig_e = numpyro.sample("sig_e", dist.Uniform(0.05, 0.5))
    sigma_sing = numpyro.sample("sigma_sing", dist.Uniform(0.01, 0.5))
    # Population log10(P).  Truth (Raghavan+2010) underlying Normal
    # mean (5.03) is well above the truncation upper bound (3), so
    # the data only constrains the TRUNCATED shape -- (mu, sig) are
    # weakly identifiable individually.  Wide priors here let the
    # posterior find the degeneracy.
    mu_logP = numpyro.sample("mu_logP", dist.Uniform(-2.0, 8.0))
    sig_logP = numpyro.sample("sig_logP", dist.Uniform(0.1, 5.0))

    log_M2 = jnp.log(M2_pad_j)

    # Binary branch
    trunc_logM2 = dist.TruncatedNormal(
        mu_logM2, sig_logM2, low=LOG_M2_LOW, high=LOG_M2_HIGH
    )
    log_pop_M2_bin = trunc_logM2.log_prob(log_M2) - log_M2
    log_pop_M2_bin = jnp.where(
        (log_M2 > LOG_M2_LOW) & (log_M2 < LOG_M2_HIGH),
        log_pop_M2_bin,
        -jnp.inf,
    )
    log_pop_e_bin = dist.TruncatedNormal(mu_e, sig_e, low=0.0, high=1.0).log_prob(
        e_pad_j
    )
    # NEW: period factor from TruncN(log10 P) population.
    log_pop_logP = dist.TruncatedNormal(
        mu_logP,
        sig_logP,
        low=LOG10_P_CUT_LOW,
        high=LOG10_P_CUT_HIGH,
    ).log_prob(log10_P_pad)
    # TruncN already returns -inf outside, but be explicit so out-of-cut
    # samples can't leak into log_w via NaN.
    log_pop_logP = jnp.where(in_P_cut_pad, log_pop_logP, -jnp.inf)
    log_period_factor = log_pop_logP + LOG10_PRIOR_RATIO_CONST

    log_w_bin = (
        jnp.log(f) + log_pop_M2_bin + log_pop_e_bin - log_pint_e_pad + log_period_factor
    )

    # Single branch
    log_pop_M2_sing = dist.HalfNormal(sigma_sing).log_prob(M2_pad_j)
    log_w_sing = jnp.log1p(-f) + log_pop_M2_sing

    log_w_mixed = jnp.logaddexp(log_w_bin, log_w_sing)
    log_w = log_w_mixed + log_dM2_dK_pad - log_pint_K_pad

    log_w = jnp.where(mask_j_5, log_w, -jnp.inf)
    log_bar_w = jax.scipy.special.logsumexp(log_w, axis=1) - ln_J_n_5
    log_bar_w = jnp.where(jnp.isfinite(log_bar_w), log_bar_w, -jnp.inf)
    numpyro.factor("loglik", log_bar_w.sum())


kernel = NUTS(population_model)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=1000, num_chains=2, progress_bar=True)
mcmc.run(jr.key(0))
post5 = mcmc.get_samples()
mcmc.print_summary()

In [ ]:
# 5.3 Corner plot of the hyperparameter posterior with truth markers.

import corner

truth_f = float(truths["binary_fraction_true"])
_log_M2_grid_t = np.log(np.geomspace(0.1, 0.9, 10000))
truth_mu_logM2 = float(np.mean(_log_M2_grid_t))
truth_sig_logM2 = float(np.std(_log_M2_grid_t))
truth_mu_e = float(truths["ecc_dist_true"][0])
truth_sig_e = float(truths["ecc_dist_true"][1])
truth_mu_logP = float(truths["log10P_mean_true"])  # underlying Normal mean
truth_sig_logP = float(truths["log10P_std_true"])  # underlying Normal stddev

print("Truth markers:")
print(f"  f          = {truth_f:.3f}")
print(f"  mu_logM2   ~ {truth_mu_logM2:.3f}    (mean log of LogUniform[0.1, 0.9])")
print(f"  sig_logM2  ~ {truth_sig_logM2:.3f}    (stddev log of LogUniform[0.1, 0.9])")
print(f"  mu_e       = {truth_mu_e:.3f}")
print(f"  sig_e      = {truth_sig_e:.3f}")
print(f"  mu_logP    = {truth_mu_logP:.3f}    (Raghavan+2010, log10 P [day])")
print(f"  sig_logP   = {truth_sig_logP:.3f}    (Raghavan+2010, log10 P [day])")
print("  sigma_sing : no truth value (single-component width is a model parameter)")

var_names = [
    "f",
    "mu_logM2",
    "sig_logM2",
    "mu_e",
    "sig_e",
    "mu_logP",
    "sig_logP",
    "sigma_sing",
]
labels = [
    r"$f$",
    r"$\mu_{\log M_2}$",
    r"$\sigma_{\log M_2}$",
    r"$\mu_e$",
    r"$\sigma_e$",
    r"$\mu_{\log_{10} P}$",
    r"$\sigma_{\log_{10} P}$",
    r"$\sigma_{\rm sing}$",
]
samples_arr = np.column_stack([np.asarray(post5[name]) for name in var_names])
truths_arr = [
    truth_f,
    truth_mu_logM2,
    truth_sig_logM2,
    truth_mu_e,
    truth_sig_e,
    truth_mu_logP,
    truth_sig_logP,
    None,
]

fig = corner.corner(
    samples_arr,
    labels=labels,
    truths=truths_arr,
    truth_color="C1",
    show_titles=True,
    title_kwargs={"fontsize": 9},
    label_kwargs={"fontsize": 10},
    hist_kwargs={"density": True},
)
fig.set_size_inches(12, 12)

In [ ]:
# 5.4 Posterior-predictive overlays for p(M2) and p(e).

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# p(M2): doubly-truncated LogNormal posterior band vs truth LogUniform.
M2_grid = np.geomspace(0.02, 5.0, 200)
log_M2_grid = np.log(M2_grid)
n_curves = 200
idx_pp = np.random.default_rng(0).choice(
    post5["f"].shape[0], size=n_curves, replace=False
)
ln_curves_M2 = np.full((n_curves, M2_grid.size), -np.inf)
for i, k in enumerate(idx_pp):
    mu_k = float(post5["mu_logM2"][k])
    sig_k = float(post5["sig_logM2"][k])
    log_dens = np.asarray(
        dist.TruncatedNormal(
            mu_k, sig_k, low=float(LOG_M2_LOW), high=float(LOG_M2_HIGH)
        ).log_prob(jnp.asarray(log_M2_grid))
    )
    inside = (log_M2_grid > float(LOG_M2_LOW)) & (log_M2_grid < float(LOG_M2_HIGH))
    ln_curves_M2[i, inside] = (log_dens - log_M2_grid)[inside]
pdf_curves = np.exp(ln_curves_M2)
lo, med, hi = np.percentile(pdf_curves, [16, 50, 84], axis=0)
axes[0].plot(M2_grid, med, "C0-", lw=2, label="median (Section 5)")
axes[0].fill_between(M2_grid, lo, hi, color="C0", alpha=0.25, label="16-84% band")
M2_min_t, M2_max_t = 0.1, 0.9
trupd = np.where(
    (M2_grid >= M2_min_t) & (M2_grid <= M2_max_t),
    1.0 / (M2_grid * np.log(M2_max_t / M2_min_t)),
    0.0,
)
axes[0].plot(M2_grid, trupd, "C2-", lw=1.5, label="truth LogUniform[0.1, 0.9]")
axes[0].set_xscale("log")
axes[0].set_xlabel("$M_2$ [$M_\\odot$]")
axes[0].set_ylabel("$p(M_2)$")
axes[0].legend(fontsize=8)
axes[0].axvline(0.1, color="grey", ls=":", lw=1)

# p(e)
e_grid = np.linspace(1e-3, 1 - 1e-3, 200)
ln_curves_e = np.empty((n_curves, e_grid.size))
for i, k in enumerate(idx_pp):
    ln_curves_e[i] = np.asarray(
        dist.TruncatedNormal(
            post5["mu_e"][k], post5["sig_e"][k], low=0.0, high=1.0
        ).log_prob(jnp.asarray(e_grid))
    )
lo, med, hi = np.percentile(ln_curves_e, [16, 50, 84], axis=0)
axes[1].plot(e_grid, np.exp(med), "C0-", lw=2, label="median TruncN")
axes[1].fill_between(e_grid, np.exp(lo), np.exp(hi), color="C0", alpha=0.25)
axes[1].plot(
    e_grid,
    np.exp(
        np.asarray(
            dist.TruncatedNormal(truth_mu_e, truth_sig_e, low=0.0, high=1.0).log_prob(
                jnp.asarray(e_grid)
            )
        )
    ),
    "C2-",
    lw=1.5,
    label=f"truth TruncN({truth_mu_e}, {truth_sig_e})",
)
axes[1].plot(
    e_grid,
    np.exp(
        np.asarray(prior.nonlinear_priors["eccentricity"].log_prob(jnp.asarray(e_grid)))
    ),
    "k--",
    lw=1,
    label="interim prior",
)
axes[1].set_xlabel("eccentricity $e$")
axes[1].set_ylabel("$p(e)$")
axes[1].legend(fontsize=8)
fig.tight_layout()

In [ ]:
# 5.5 Close-binary fraction posterior.
#
# f is the fraction of stars drawn from the binary component (rather than
# the HalfNormal single component).  Multiply by the mass of the binary
# M_2 distribution above 0.1 Msun (truncated at 0.05) to match Section 4.

M2_LOWER_COMPARE = 0.1


def _f_close_ratio(mu, sig):
    base = dist.TruncatedNormal(mu, sig, low=float(LOG_M2_LOW), high=float(LOG_M2_HIGH))
    log_above_low = jnp.log1p(-base.cdf(float(LOG_M2_LOW)))
    log_above_cmp = jnp.log1p(-base.cdf(jnp.log(M2_LOWER_COMPARE)))
    return jnp.exp(log_above_cmp - log_above_low)


ratio = jax.vmap(_f_close_ratio)(post5["mu_logM2"], post5["sig_logM2"])
f_close = np.asarray(post5["f"]) * np.asarray(ratio)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(
    f_close,
    bins=40,
    density=True,
    alpha=0.7,
    color="C0",
    label=r"Section 5: $f_\mathrm{close} = f \cdot P(M_2 > 0.1)$",
)
ax.axvline(f_hat, color="C3", lw=2, ls="--", label=f"Section 4 MAP = {f_hat:.3f}")
ax.axvline(truth_f, color="C2", lw=2, label=f"truth = {truth_f:.3f}")
ax.set_xlabel("close binary fraction")
ax.set_ylabel("posterior density")
ax.legend()
fig.tight_layout()

lo_raw, med_raw, hi_raw = np.percentile(np.asarray(post5["f"]), [16, 50, 84])
lo, med, hi = np.percentile(f_close, [16, 50, 84])
print("Section 5 raw  f (binary fraction across the inferred M2 support):")
print(f"    f = {med_raw:.3f}  (-{med_raw - lo_raw:.3f}, +{hi_raw - med_raw:.3f})")
print("Section 5      close (M_2 > 0.1):")
print(f"    f = {med:.3f}  (-{med - lo:.3f}, +{hi - med:.3f})")
print(f"Section 4 hard-cut MAP:             f = {f_hat:.3f}")
print(f"Truth:                              f = {truth_f:.3f}")

### Notes and open knobs

- **Period distribution.** We held $p(P)$ fixed at the LogUniform interim prior. The simulator's truth is LogUniform on a sub-range, so the period factor approximately cancels and inferring $\boldsymbol\alpha_P$ jointly with the rest would mostly add a noise floor. A real application (especially over Öpik's-law-violating regimes like compact-binary populations) would want to extend the model with a LogNormal $p(P)$ and compute the corresponding interim-prior renormaliser inside the MCMC.
- **Per-star $(M_1, \sin i)$ jitter.** We use a single $(M_1, \sin i)$ draw per posterior sample, so $M_2^{(nj)}$ is one Monte-Carlo realisation of the true $M_2$ posterior. Drawing multiple $(M_1, \sin i)$ pairs per posterior sample (then averaging the weight over the auxiliaries) would tighten $\hat f$ marginally; not a tutorial-blocker.
- **"Single" model.** Stars in the "single" component are treated as drawn from the rejection-sampler interim prior — i.e. their posterior is uninformative about the population. A stricter $K = 0$ alternative would change $p_\mathrm{sing}(D_n)$ but in our tests gives consistent $\hat f$ within the credible region.
- **Section 4 contrast.** Section 4's biased $\hat f$ (vertical dashed line in the previous figure) sits below the Section 5 16% percentile, which is the expected sign and magnitude of the boundary-cut bias.

## Caveats and TODOs

### Variable-epoch stars

This tutorial assumes every star has the same number of epochs (`N_epochs`). In real surveys that's rarely true. The current `RejectionSampler` JIT cache is keyed on the shape of `data.time` / `data.rv` / `data.rv_err`, so a heterogeneous epoch-count population would trigger one JIT compile per distinct epoch count. Options to explore:

1. **Bucket by epoch count.** Group stars by $N_\mathrm{epochs}$ and process each bucket in turn; the cache is reused within each bucket. Simple, no API changes.
2. **Pad + mask.** Pad every dataset to a common $N_\max$ and pass a boolean `epoch_mask` through `data`. Requires `RVData` to gain a mask-aware likelihood (an extension or a new `Masked` wrapper). The likelihood would zero out the contribution of padded rows.
3. **vmap over stars.** With padded shapes, `jax.vmap` the whole `run_with_samples` pipeline so the entire population is one XLA program. Highest performance ceiling but biggest engineering lift.

Bucketing (1) is the cheapest win and likely sufficient for most real surveys.

### Primary-mass distribution

We modelled $M_1$ as a Gaussian per star with a placeholder $\sigma_{M_1}$. In practice you'd pull the per-star mean and standard deviation from an external catalogue. The hierarchical step (Section 4) is unchanged — only the $M_1$ draws used to compute $M_{2, nj}$ change.

### DWH's objection in the project note

DWH points out that even with arbitrarily good RV data per star, you only ever measure $M_2 \sin i$, so a clean cut on $M_2$ requires hierarchical structure even in the high-data limit. The procedure in Section 4 already does the right Monte-Carlo over $\sin i$, so $\hat f$ here is well-defined; refining it to a *joint* hierarchical inference over $(f, p(M_2), p(\sin i))$ is a clean follow-up.

### v_sys refinement (optional)

The note also mentions using $v_0$ to refine membership selection. Easy to add: just inspect `posteriors[n].linear["v_sys"]` and reject stars whose posterior is inconsistent with the cluster mean.
